### Plotting the development of aligned communities over years

Community nodes are aligned using a "working graph".
The working graph is based on the community evolution graph. Edges are removed if they have a weight of at most 25%, 20%, 15% of the ingoing and of the outgoing edge.
For this diagram we use connected components of this working graph and name the connected compoents by the larges single node in each component.

In [ ]:
import sys
sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
%run common.py

import matplotlib.patches as patches
import numpy as np
import regex
import json
import pandas as pd
from collections import Counter, defaultdict
from scipy import stats
from matplotlib.legend import Legend
import matplotlib.pyplot as plt
import functools
import seaborn as sns

import matplotlib.pyplot as plt
plt.style.use('altair.mplstyle')

plt.rcParams['axes.formatter.limits'] = (-10, 10) # for consistently non-scientific notation
plt.rcParams.update({'font.size': 14})

In [ ]:
from matplotlib.colors import ListedColormap
import seaborn as sns

def cluster_family_colors(dataset, n_colors=20, graycolor=False):
    if graycolor:
        greyscale_palette = sns.dark_palette("#ccc", n_colors=n_colors)
        
        half_size = n_colors // 2
        first_half = greyscale_palette[:half_size]
        second_half = greyscale_palette[half_size:]

        zippered_palette = [val for pair in zip(first_half, second_half) for val in pair]

        if len(second_half) > len(first_half):
            zippered_palette.append(even_colors[-1])
        return zippered_palette
    return sns.color_palette("tab20", n_colors=n_colors)

In [ ]:
def get_manual_labels(path):
    df = pd.read_csv(path)
    return {cluster: label for cluster, label in zip(df['Cluster ID'], df['Label'])}

In [ ]:
from matplotlib.legend import Legend
import functools
    
def subtitle_decorator(handler):
    @functools.wraps(handler)
    def wrapper(legend, orig_handle, fontsize, handlebox):
        handle_marker = handler(legend, orig_handle, fontsize, handlebox)
        if handle_marker.get_alpha() == 0:
            handlebox.set_visible(False)
    return wrapper

#Adds our decorator to all legend handler functions
for handler in Legend.get_default_handler_map().values():
    handler.legend_artist = subtitle_decorator(handler.legend_artist)

In [ ]:
def filter_edges(G, threshold):
    edges_to_remove = [
        (u, v) 
        for u, v, data in G.edges(data=True) 
        if (
            data['weight'] < G.nodes[u]['weight'] * threshold or
            data['weight'] < G.nodes[v]['weight'] * threshold 
        )
    ]
    H = G.copy()
    H.remove_edges_from(edges_to_remove)
    return H


def count_weight_per_year(nodes_set, H):
    counts = defaultdict(int)
    for node in nodes_set:
        year = node.split('_')[0]
        counts[year] += H.nodes[node]['weight']
    return dict(counts)


def get_connected_components_per_year(H):
    components = list(nx.connected_components(H.to_undirected()))
    components.sort(
        key=lambda nodes_set: (max([H.nodes[n]["tokens_n"] for n in nodes_set]), sorted(nodes_set)[-1]),
        reverse=True
    )
    component_weight_per_year = [
        count_weight_per_year(nodes_set, H)
        for nodes_set in components
    ]
    return component_weight_per_year, components

def get_linear_regression_df(df):
    linreg = [(
        label, 
        stats.linregress(range(len(row)), row.fillna(0).to_list())
    ) for label, row in df.T.iterrows()]
    
    number_of_years = len(df.T.columns)

    linreg_df = pd.DataFrame([
        dict(
            label=label, 
            intercept=d.intercept, 
            slope=d.slope, 
            rvalue=d.rvalue, 
            r2value=d.rvalue**2, 
            pvalue=d.pvalue, 
            stderr=d.stderr
        ) 
        for label, d in linreg
    ])
    linreg_df = linreg_df.set_index('label')
    pd.set_option('display.float_format', lambda x: '%.2f' % x)
    linreg_df['size_mean'] =  df.fillna(0).mean()
    linreg_df['slope_rel'] = linreg_df.slope / linreg_df.size_mean
    linreg_df['last_year_value'] = linreg_df.intercept + linreg_df.slope * (number_of_years - 1)
    return linreg_df

In [ ]:
def format_func(value, tick_number):
        if value == 0:
            return '0,0'
        return '{:.1e}'.format(value).replace('e+0', 'e').replace(',', ' ').replace('.', ',')
    

def plot_community_development(
    config_str, dataset, threshold, name, 
    selected_clusters=10, colors=None, 
    most_growing_labels=None, hue_order=None,
    show_manual_labels=True,
    graycolor=False
):
    # Load Graph
    global G
    G = nx.read_gpickle(
        path=f'../../legal-networks-data/{dataset.lower()}/13_cluster_evolution_graph/all_{config_str}.gpickle.gz',
    )

    # Select weights
    nx.set_node_attributes(G, nx.get_node_attributes(G, 'tokens_n'), 'weight')
    nx.set_edge_attributes(G, nx.get_edge_attributes(G, 'tokens_n'), 'weight')
    
    H = filter_edges(G, threshold)
    

    component_weight_per_year, components = get_connected_components_per_year(H)

    # Format numbers
    df = pd.DataFrame(component_weight_per_year).T.sort_index()
    df.columns = [
        sorted(nodes_set, key=lambda n: (H.nodes[n]["tokens_n"], n))[-1]
        for nodes_set in components
    ]
    
    # Setup cluster filter
    if type(selected_clusters) is int:
        selected_clusters = df.columns[:selected_clusters]
        selected_cluster_labels = None
    elif type(selected_clusters) is dict:
        selected_cluster_labels = selected_clusters
        selected_clusters = list(selected_clusters.keys())
    elif type(selected_clusters) is list:
        selected_cluster_labels = None
    else:
        raise Exception('Wrong selected_clusters type')
    try:
        df_selected = df.loc[:,selected_clusters]
    except KeyError :
        print('Missing:', sorted(set(selected_clusters) - set(df.columns)))
        raise
        
    # Calculate growth of selected clusters
    growth_selected = df_selected.iloc[-1].sum() - df_selected.iloc[0].sum()
    print('Selected growth (abs.):', growth_selected)
    growth_total = df.iloc[-1].sum() - df.iloc[0].sum()
    print('Total growth (abs.):', growth_total)
    print('Selected growth (rel.):', growth_selected/growth_total)
    
    selected_size_ratio_df = df_selected.T.sum() / df.T.sum()
    print('Selected sizes per year statistics:')
    print(selected_size_ratio_df.describe())
    
    # Convert data into long format (for seaborn)
    global df_melt
    df_melt = pd.melt(df_selected.reset_index(), id_vars=['index'])
    df_melt.columns = ['Year', 'Cluster', 'Tokens']
    
    # Hue order
    leading_clusters = [c[0] for c in cluster_families(G, 0.15)]
    if not hue_order:
        hue_order = sorted(
            df_melt.Cluster.unique(),
            key=lambda x: leading_clusters.index(x)
        )

    # Plot scatter
    filled_markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', 'h', 'H', '*', 'P', 'X']  # Filled markers
    plt.figure(figsize=(9, 6))
    ax = sns.scatterplot(
        x='Year', 
        y='Tokens', 
        hue='Cluster',
        hue_order=hue_order,
        data=df_melt, 
        palette=cluster_family_colors(dataset, len(df_selected.columns), graycolor) if colors is None else colors,
        style='Cluster',
        markers={key: filled_markers[i % len(filled_markers)] for i, key in enumerate(df_melt['Cluster'].unique())},
        s=80
    )
    ax.set_xticklabels(df.T.columns, rotation=90)
    
    # Regression
    linreg_df = get_linear_regression_df(df)
    line_df = pd.DataFrame([
        {
            'label': label,
            df.T.columns[0]: row.intercept,
            df.T.columns[-1]: row.intercept + row.slope * (len(df.T.columns) - 1)
        }
        for label, row in linreg_df.iterrows()
    ])
    line_df = line_df.set_index('label')
    
    line_df_selected = line_df.loc[selected_clusters]
    line_df_selected.index.names = ['label']
    
    # Convert data into long format (for seaborn)
    df_line_melt = pd.melt(line_df_selected.reset_index(), id_vars=['label'])
    df_line_melt.columns = ['Cluster', 'Year', 'Tokens']
        
    # Plot lines
    sns.lineplot(
        x='Year', 
        y='Tokens', 
        hue='Cluster', 
        hue_order=hue_order,
        data=df_line_melt, 
        palette=cluster_family_colors(dataset, len(df_selected.columns), graycolor) if colors is None else colors,
        legend=False,
    )
        
    handles, labels = ax.get_legend_handles_labels()
    handle_dict = {label: handle for handle, label in zip(handles, labels)}
    labels = line_df_selected.sort_values(line_df_selected.columns[-1], ascending=False).index.to_list()
    handles = [handle_dict[l] for l in labels]
    for handle in handles:
        handle._sizes = [80]
    
    labels = [f'{l.split("_")[-1]} in {l.split("_")[0][:4]}' for l in labels]
    if show_manual_labels:
        manual_labels = get_manual_labels(f'../labels-{dataset.lower()}-{config_str}.csv')
        labels = [
            f'{manual_labels[l]} – {l}' if l in manual_labels else l
            for l in labels
        ]
    
    legend = ax.legend(handles, labels, loc='upper center', 
                bbox_to_anchor=(0.5, -0.15), frameon=False, ncol=(1 if show_manual_labels else 4))
    ax.set_xticklabels(range(1994,2019+1))
    plt.ylim(0,df_line_melt.Tokens.max()*1.2) 

    ax.yaxis.set_major_formatter(plt.FuncFormatter(format_func))
    
    plt.savefig(
        f'../data_figures/cluster-evolution-size-dynamics-{dataset.lower()}-{config_str}-{name}{"_graycolor" if graycolor else ""}.pdf', 
        bbox_extra_artists=(legend,), 
        bbox_inches='tight'
    )
    
    linreg_df_selected = linreg_df.loc[selected_clusters]
    return linreg_df_selected, components, hue_order

In [ ]:
def analyse_slope_to_size(linreg_df_top, dataset, name):
    plt.figure(figsize=(9, 6))
    regplt = sns.regplot(linreg_df_top.slope, linreg_df_top.size_mean)
    regplt.set(
        ylim=(0, None),
        xlabel='Slope',
        ylabel='Mean size'
    )
    regplt.xaxis.set_major_formatter(plt.FuncFormatter(format_func))
    regplt.yaxis.set_major_formatter(plt.FuncFormatter(format_func))


    print(stats.linregress(
        linreg_df_top.slope.fillna(0), 
        linreg_df_top.size_mean.fillna(0),
    ))
    plt.savefig(
        f'../data_figures/cluster-evolution-slope-size-regression-{dataset.lower()}-{name}.pdf', 
    )
    

def analyse_slope_rel_to_size(linreg_df_top):
    ax = sns.regplot(linreg_df_top.slope_rel, linreg_df_top.size_mean)
    print(stats.linregress(
        linreg_df_top.slope_rel.fillna(0), 
        linreg_df_top.size_mean.fillna(0)
    ))

In [ ]:
def format_linreg_df(df, country_code=None):
    df = df.sort_values('last_year_value', ascending=False)
    df.index = [l.split('– ', 1)[-1] for l in df.index]
    df = df.reset_index()
    df = df[[
        'index', 
        'slope', 
        'intercept', 
        'pvalue', 
        'rvalue',  
        'r2value',  
        'stderr', 
        'size_mean'
    ]]
    df.columns = [
        'Gruppenfamilie', 
        'Steigung', 
        'Achsenabschnitt', 
        'P-Wert', 
        '$r$', 
        '$r^2$', 
        'Standardfehler', 
        'Durchschnitt'
    ]
    df = df.drop('P-Wert', axis=1)
    df.Gruppenfamilie = [n.split('_')[1] + ' in ' + n[:4] for n in df.Gruppenfamilie]
    if country_code:
        df['Land'] = country_code
        df = df.set_index(['Land', 'Gruppenfamilie'])
    else: 
        df = df.set_index(['Gruppenfamilie'])
    return df.sort_values('Steigung', ascending=False)

## Run DE

### Top 20 for inspection

In [ ]:
config_str='0-0_1-0_-1_o-2-0_t-paragraph_a-infomap_n100_m1-0_s0_c1000'
linreg_df_de, components, hue_order_de = plot_community_development(
    config_str, 
    dataset='de', 
    threshold=0.15,
    selected_clusters=20,
    name='top20'
)

In [ ]:
config_str='0-0_1-0_-1_o-2-0_t-paragraph_a-infomap_n100_m1-0_s0_c1000'
linreg_df_de, components, hue_order_de = plot_community_development(
    config_str, 
    dataset='de', 
    threshold=0.15,
    selected_clusters=20,
    name='top20',
    graycolor=True
)

In [ ]:
analyse_slope_to_size(linreg_df_de, 'de', 'top20')

In [ ]:
df = format_linreg_df(linreg_df_de, country_code=None).reset_index()
latex_str = df.to_latex(index=False, escape=False)
latex_str = latex_str.replace(
    "Gruppenfamilie &  Steigung &  Achsenabschnitt &   $r$ &  $r^2$ &  Standardfehler &  Durchschnitt",
    "\\makecell{Gruppen- \\\\ familie} &  Steigung &  \\makecell{Achsen- \\\\ abschnitt} &   $r$ &  $r^2$ &  \\makecell{Standard- \\\\ fehler} &  \\makecell{Durch- \\\\ schnitt}  \\"
)
assert latex_str.find("\\makecell{")
with open(f'../tables/meso_regression_de_{config_str}.tex', 'w') as f:
    f.write(latex_str)

In [ ]:
df

In [ ]:
config_str='0-0_1-0_-1_o-1-0_t-paragraph_a-louvain_m1-0_s0_c1000'
linreg_df_de, components, hue_order_de = plot_community_development(
    config_str, 
    dataset='de', 
    threshold=0.075,
    selected_clusters=6,
    name='top20',
)

In [ ]:
config_str='0-0_1-0_-1_o-1-0_t-paragraph_a-louvain_m1-0_s0_c1000'
linreg_df_de, components, hue_order_de = plot_community_development(
    config_str, 
    dataset='de', 
    threshold=0.075,
    selected_clusters=6,
    name='top20',
    graycolor=True
)

In [ ]:
df = format_linreg_df(linreg_df_de, country_code=None).reset_index()
with open(f'../tables/meso_regression_de_{config_str}.tex', 'w') as f:
    df.to_latex(f, index=False, escape=False)

In [ ]:
df